## Documentation

To read more about the get API, visit the [docs](https://www.elastic.co/guide/en/elasticsearch/reference/current/docs-get.html).

![get_documents_docs](../images/get_documents_docs.png)

## Connect to ElasticSearch

In [1]:
from pprint import pprint
from elasticsearch import Elasticsearch
from es_config import es

client_info = es.info()
print('Connected to Elasticsearch!')
pprint(client_info.body)

Connected to Elasticsearch!
{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'dH7SsBWKSeS7pB6Dceg5gw',
 'name': 'ad5294c1e0c0',
 'tagline': 'You Know, for Search',
 'version': {'build_date': '2026-09-09T22:42:53.976833287Z',
             'build_flavor': 'default',
             'build_hash': '9170df19cae1adb107b7b489b4d82dec66d7a337',
             'build_snapshot': False,
             'build_type': 'docker',
             'lucene_version': '10.5.1',
             'minimum_index_compatibility_version': '8.0.0',
             'minimum_wire_compatibility_version': '8.19.0',
             'number': '9.5.4'}}


c:\Users\rgak2\Desktop\elasticSearch\elastic_search\Lib\site-packages\urllib3\connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## Index documents

In [2]:
es.indices.delete(index='my_index', ignore_unavailable=True)
es.indices.create(index='my_index')

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_index'})

We are using the `dummy_data.json` file to insert multiple documents in the index. We store the id of each document in the `document_ids` list.

In [37]:
import json
from tqdm import tqdm


document_ids = []
dummy_data = json.load(open("../data/dummy_data.json"))
for document in tqdm(dummy_data, total=len(dummy_data)):
    response = es.index(index='my_index', body=document)
    document_ids.append(response['_id'])

100%|██████████| 3/3 [00:00<00:00, 18.01it/s]


In [38]:
document_ids

['v3jeJaEBXbwmI36BnXEF', 'wHjeJaEBXbwmI36BnXFA', 'wXjeJaEBXbwmI36BnXF0']

## Get operation

This is an example of a successful operation. If the `ID` exists in the document, the `get` operation won't return any errors.

In [39]:
try:
    response = es.get(index='my_index', id=document_ids[2])
except Exception as e:
    print(e)

In [40]:
from pprint import pprint

pprint(response.body)

{'_id': 'wXjeJaEBXbwmI36BnXF0',
 '_index': 'my_index',
 '_primary_term': 1,
 '_seq_no': 12,
 '_source': {'created_on': '2024-09-24',
             'text': 'The content of the third document goes here.',
             'title': 'Sample Title 3'},
 '_version': 1,
 'found': True}


This example shows that the `get` operation fails when providing it with an `ID` that does not exist in the index.

In [41]:
try:
    response = es.get(index='my_index', id="id")
except Exception as e:
    print(e)

NotFoundError(404, "{'_index': 'my_index', '_id': 'id', 'found': False}")
